<h1>Chapter 8 - Agentic RAG</h1>
<i>Building an agentic system using function calling without a framework.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch08_agentic_rag/8.4_building_agentic_system_function_calling/building_agents_without_framework.ipynb)

---

This notebook is for Chapter 8 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


### Install Required Packages

In [9]:
!pip install openai

  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Usi

### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the colab secrets and load it to the environmental variables.

In [1]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

# Building an Agentic System that can Calculate Using Function Calling

This notebook mirrors the first agent recipe from your *Agentic RAG* chapter, where you build an agent **without a framework** using classic function calling.

You can upload this notebook directly to **Google Colab**.  
Before running the examples, make sure you have a valid **OpenAI API key** available as `OPENAI_API_KEY`.


In [7]:
def add_numbers(a, b):
    """
    Tool for adding two numbers together.

    This tool takes two numbers as input and returns their sum. It can be
    used for:
    - Basic addition operations
    - Combining numeric values
    - First step in more complex calculations

    Args:
        a (float): The first number to add
        b (float): The second number to add

    Returns:
        float: The sum of the two input numbers (a + b)

    Example:
        add_numbers(5.0, 3.0) -> 8.0
        add_numbers(-1.0, 1.0) -> 0.0
    """
    return a + b


def multiply_numbers(a, b):
    """
    Tool for multiplying two numbers together.

    This tool takes two numbers as input and returns their product. It can be
    used for:
    - Basic multiplication operations
    - Scaling numeric values
    - Completing complex calculations

    Args:
        a (float): The first number to multiply
        b (float): The second number to multiply

    Returns:
        float: The product of the two input numbers (a * b)

    Example:
        multiply_numbers(5.0, 3.0) -> 15.0
        multiply_numbers(-1.0, 2.0) -> -2.0
    """
    return a * b


In [8]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "add_numbers",
      "description": add_numbers.__doc__,
      "parameters": {
        "type": "object",
        "properties": {
          "a": {
            "type": "number",
            "description": "The first number to add"
          },
          "b": {
            "type": "number",
            "description": "The second number to add"
          }
        },
        "required": ["a", "b"]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "multiply_numbers",
      "description": multiply_numbers.__doc__,
      "parameters": {
        "type": "object",
        "properties": {
          "a": {
            "type": "number",
            "description": "The first number to multiply"
          },
          "b": {
            "type": "number",
            "description": "The second number to multiply"
          }
        },
        "required": ["a", "b"]
      }
    }
  }
]


In [9]:
import openai
import json

user_query = "What is the result of (5.0 + 3.0) * 2.0?"
messages = [{"role": "user", "content": user_query}]

# Call the LLM with the available tools
response = openai.chat.completions.create(
    model="gpt-5-mini", messages=messages, tools=tools
)

# --- Extraction and Execution Logic ---
message = response.choices[0].message
tool_calls = message.tool_calls
finish_reason = response.choices[0].finish_reason

for tool_call in tool_calls:
    tool_name = tool_call.function.name
    # 1. Extract and parse arguments
    arguments = json.loads(tool_call.function.arguments)

    # 2. Locate the function object
    tool = globals().get(tool_name)

    # 3. Execute the function and capture the result
    result = tool(**arguments) if tool else {}


In [13]:
def process_function_invocations(invocations):
    """Processes LLM tool requests and executes the functions."""
    results = []
    for invocation in invocations:
        function_name = invocation.function.name
        function_args = json.loads(invocation.function.arguments)

        # Get and execute the function
        function = globals().get(function_name)
        # Execute the function with unpacked arguments
        function_result = function(**function_args) if function else {}

        # Format the result to be sent back to the LLM
        results.append({
            "role": "tool",
            "content": json.dumps(function_result), # Must be a string
            "tool_call_id": invocation.id,
        })
    return results

user_query = "What is the result of (6.7 + 3.3) * 2.0?"
messages = [{"role": "user", "content": user_query}]
finish_reason = "tool_calls" # Initialize to enter the loop

while finish_reason == "tool_calls":
    # Call the LLM with the conversation history and tools
    response = openai.chat.completions.create(
        model="gpt-5-mini",
        messages=messages,
        tools=tools
    )

    # Extract response details
    message = response.choices[0].message
    tool_calls = message.tool_calls
    finish_reason = response.choices[0].finish_reason

    # Add the LLM's response (tool request or final answer) to history
    messages.append(message)

    if finish_reason == "tool_calls":
        # Execute the tool and format the results for feedback
        tool_results = process_function_invocations(tool_calls)
        # Add the tool results to the history
        messages.extend(tool_results)

print(messages)


[{'role': 'user', 'content': 'What is the result of (6.7 + 3.3) * 2.0?'}, ChatCompletionMessage(content='(6.7 + 3.3) = 10.0, and 10.0 * 2.0 = 20.0.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)]


In [14]:
user_query = (
    "I'm planning a trip to London. "
    "I'll arrive in 5 days and stay for 6 days. "
    "Can you tell me what the current weather is in London, "
    "and also calculate in how many days I will return home from today?"
)

messages = [{"role": "user", "content": user_query}]

finish_reason = "tool_calls"  # Initialize finish_reason

while finish_reason == "tool_calls":
    print("messages:", messages)
    response = openai.chat.completions.create(
        model="gpt-5-mini",
        messages=messages,
        tools=tools
    )

    finish_reason = response.choices[0].finish_reason

    if finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = process_function_invocations(tool_calls)

        messages.append(message)
        messages.extend(results)

final_content = response.choices[0].message.content
print("Final content:", final_content)


messages: [{'role': 'user', 'content': "I'm planning a trip to London. I'll arrive in 5 days and stay for 6 days. Can you tell me what the current weather is in London, and also calculate in how many days I will return home from today?"}]


Final content: I can’t access live internet data, so I can’t fetch the current real-time weather in London right now. If you’d like, I can tell you how to check it quickly (e.g., BBC Weather, Met Office, Weather.com, or “London weather” in Google or your phone’s weather app), or you can tell me a weather service you prefer and I’ll give step-by-step instructions for that.

If you want a quick expectation instead of real-time data: historically for late February / early March in London expect cool weather — daytime highs around 7–10°C (45–50°F), nights around 2–5°C (35–40°F), and a moderate chance of rain or drizzle and occasional wind. (This is climatological guidance, not the current conditions.)

Return-date calculation
- Today (current date) is 2026-02-24.
- You arrive in 5 days and then stay 6 days, so days until return = 5 + 6 = 11 days from today.
- 11 days from 2026-02-24 is 2026-03-07.

So you will return home in 11 days, on March 7, 2026.

Would you like me to check a specific

In [17]:
import requests

def get_weather(latitude, longitude):
    """
    Tool for getting weather data for a specific location.

    This tool takes latitude and longitude coordinates as input and returns
    weather information for that location. It can be used for:
    - Retrieving current weather conditions
    - Getting temperature and weather forecasts
    - Checking weather at specific geographic coordinates

    Args:
        latitude (float): The latitude of the location to get weather data for
        longitude (float): The longitude of the location to get weather data for

    Returns:
        dict: Weather information for the specified location including
              temperature, conditions, and other relevant weather data

    Example:
        get_weather(51.5074, -0.1278) -> Weather data for London
        get_weather(40.7128, -74.0060) -> Weather data for New York
    """
    try:
        # Call Open-Meteo API for current weather
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code"
        }
        
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        current = data.get("current", {})
        
        return {
            "latitude": latitude,
            "longitude": longitude,
            "temperature": current.get("temperature_2m"),
            "humidity": current.get("relative_humidity_2m"),
            "wind_speed": current.get("wind_speed_10m"),
            "weather_code": current.get("weather_code")
        }
    except Exception as e:
        return {
            "error": str(e),
            "latitude": latitude,
            "longitude": longitude
        }

In [18]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": add_numbers.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "number",
                        "description": "The first number to add"
                    },
                    "b": {
                        "type": "number",
                        "description": "The second number to add"
                    }
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "multiply_numbers",
            "description": multiply_numbers.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "number",
                        "description": "The first number to multiply"
                    },
                    "b": {
                        "type": "number",
                        "description": "The second number to multiply"
                    }
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": get_weather.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "The latitude of the location to get weather data for"
                    },
                    "longitude": {
                        "type": "number",
                        "description": "The longitude of the location to get weather data for"
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    }
]

In [ ]:
import openai
import json

user_query = "What is the result of (5.0 + 3.0) * 2.0?"
messages = [{"role": "user", "content": user_query}]

# Call the LLM with the available tools
response = openai.chat.completions.create(
    model="gpt-4o-mini", messages=messages, tools=tools
)

# --- Extraction and Execution Logic ---
message = response.choices[0].message
tool_calls = message.tool_calls
finish_reason = response.choices[0].finish_reason

for tool_call in tool_calls:
    tool_name = tool_call.function.name
    # 1. Extract and parse arguments
    arguments = json.loads(tool_call.function.arguments)

    # 2. Locate the function object
    tool = globals().get(tool_name)

    # 3. Execute the function and capture the result
    result = tool(**arguments) if tool else {}

In [22]:
for tool_call in tool_calls:
    print("Tool name:", tool_call.function.name)
    print("Arguments:", tool_call.function.arguments)
    print("----")

Tool name: add_numbers
Arguments: {"a":5.0,"b":3.0}
----


In [ ]:
# tag::get_tool_calls_from_response_iter[]
def process_function_invocations(invocations):
    """Processes LLM tool requests and executes the functions."""
    results = []
    for invocation in invocations:
        function_name = invocation.function.name
        function_args = json.loads(invocation.function.arguments)
        
        # Get and execute the function
        function = globals().get(function_name)
        # Execute the function with unpacked arguments
        function_result = function(**function_args) if function else {} 
        
        # Format the result to be sent back to the LLM
        results.append({
            "role": "tool",
            "content": json.dumps(function_result), # Must be a string
            "tool_call_id": invocation.id,
        })
    return results

user_query = "What is the result of (6.7 + 3.3) * 2.0?"
messages = [{"role": "user", "content": user_query}]
finish_reason = "tool_calls" # Initialize to enter the loop

while finish_reason == "tool_calls":
    # Call the LLM with the conversation history and tools
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages, 
        tools=tools
    )
    
    # Extract response details
    message = response.choices[0].message
    tool_calls = message.tool_calls
    finish_reason = response.choices[0].finish_reason
    
    # Add the LLM's response (tool request or final answer) to history
    messages.append(message)
    
    if finish_reason == "tool_calls":
        # Execute the tool and format the results for feedback
        tool_results = process_function_invocations(tool_calls)
        # Add the tool results to the history
        messages.extend(tool_results)
# end::get_tool_calls_from_response_iter[]
messages


[{'role': 'user', 'content': 'What is the result of (6.7 + 3.3) * 2.0?'},
 ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_2DNKMQDrZRATRK12xtQL7H5Q', function=Function(arguments='{"a": 6.7, "b": 3.3}', name='add_numbers'), type='function'), ChatCompletionMessageToolCall(id='call_cXWZ66wwQbta4axhReiR7yVO', function=Function(arguments='{"a": 10.0, "b": 2.0}', name='multiply_numbers'), type='function')], refusal=None, annotations=[]),
 {'role': 'tool',
  'content': '10.0',
  'tool_call_id': 'call_2DNKMQDrZRATRK12xtQL7H5Q'},
 {'role': 'tool',
  'content': '20.0',
  'tool_call_id': 'call_cXWZ66wwQbta4axhReiR7yVO'},
 ChatCompletionMessage(content='The result of \\((6.7 + 3.3) * 2.0\\) is \\(20.0\\).', role='assistant', function_call=None, tool_calls=None, refusal=None, annotations=[])]

In [ ]:
user_query = (
    "I'm planning a trip to London. "
    "I'll arrive in 5 days and stay for 6 days. "
    "Can you tell me what the current weather is in London, "
    "and also calculate in how many days I will return home from today?"
)

messages = [{"role": "user", "content": user_query}]

finish_reason = "tool_calls"  # Initialize finish_reason

while finish_reason == "tool_calls":
    print("messages:", messages)
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )

    finish_reason = response.choices[0].finish_reason

    if finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = process_function_invocations(tool_calls)

        messages.append(message)
        messages.extend(results)

final_content = response.choices[0].message.content
print("Final content:", final_content)
# end::question_about_the_weather[]

messages: [{'role': 'user', 'content': "I'm planning a trip to London. I'll arrive in 5 days and stay for 6 days. Can you tell me what the current weather is in London, and also calculate in how many days I will return home from today?"}]
messages: [{'role': 'user', 'content': "I'm planning a trip to London. I'll arrive in 5 days and stay for 6 days. Can you tell me what the current weather is in London, and also calculate in how many days I will return home from today?"}, ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_zVifN0spQgKEavjwT7VPxtjw', function=Function(arguments='{"latitude": 51.5074, "longitude": -0.1278}', name='get_weather'), type='function'), ChatCompletionMessageToolCall(id='call_hU5fdHnLA6iSvJK40MqvPKKF', function=Function(arguments='{"a": 5, "b": 6}', name='add_numbers'), type='function')], refusal=None, annotations=[]), {'role': 'tool', 'content': '5.8', 'tool_call_id': 'call_zVifN0spQgKEav